## Libraries Installation

In [1]:
!pip install azure-storage-blob
!pip install snowflake-connector-python
!pip install snowflake-sqlalchemy
!pip install pandas
!pip install requests

## Libraries Import

In [2]:
import pandas as pd
import numpy as np
import os
import re
import json
import io
import requests
import warnings
warnings.filterwarnings('ignore')
from azure.storage.blob import BlobServiceClient
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from snowflake.sqlalchemy import URL
from sqlalchemy import create_engine

## Connection String Load

In [3]:
def load_config_azure(config_path='config.json'):
    """Load Azure configuration parameters from config.json."""
    with open(config_path, 'r', encoding='utf-8') as config_file:
        config = json.load(config_file)
    return config['AZURE_CONNECTION_STRING'], config['CONTAINER_NAME']

def load_config_snowflake(config_path='config.json'):
    """Load Snowflake configuration parameters from config.json."""
    with open(config_path, 'r', encoding='utf-8') as config_file:
        config = json.load(config_file)
    return (
        config['SNOWFLAKE_USER'],
        config['SNOWFLAKE_PASSWORD'],
        config['SNOWFLAKE_ACCOUNT'],
        config['SNOWFLAKE_WAREHOUSE'],
        config['SNOWFLAKE_DATABASE'],
        config['SNOWFLAKE_SCHEMA']
    )

# Load Azure credentials
AZURE_CONNECTION_STRING, CONTAINER_NAME = load_config_azure('config.json')
blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)
print('Azure connected successfully')

# Load Snowflake credentials
SF_USER, SF_PASSWORD, SF_ACCOUNT, SF_WAREHOUSE, SF_DATABASE, SF_SCHEMA = load_config_snowflake('config.json')
engine = create_engine(URL(
    user=SF_USER,
    password=SF_PASSWORD,
    account=SF_ACCOUNT,
    warehouse=SF_WAREHOUSE,
    database=SF_DATABASE,
    schema=SF_SCHEMA
))
print('Snowflake connected successfully')

Azure connected successfully
Snowflake connected successfully


## Extraction

### Extraction - ZIP Code Population Data

In [4]:
# Extract zip.csv from Azure Blob Storage
blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob='zip.csv')
blob_data = blob_client.download_blob().readall()
zip_raw = pd.read_csv(io.BytesIO(blob_data), encoding='utf-8-sig')
zip_raw.columns = ['label', 'population']
print('ZIP raw shape:', zip_raw.shape)
zip_raw.head()

ZIP raw shape: (67548, 2)


,label,population
0,ZCTA5 00601,NaN
1,ZCTA5 00601,"17,242"
2,ZCTA5 00602,NaN
3,ZCTA5 00602,"37,548"
4,ZCTA5 00603,NaN


### Extraction - State Population Data

In [5]:
# Extract state.csv from Azure Blob Storage
blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob='state.csv')
blob_data = blob_client.download_blob().readall()
state_raw = pd.read_csv(io.BytesIO(blob_data), encoding='utf-8-sig')
state_raw.columns = ['label', 'population']
print('State raw shape:', state_raw.shape)
state_raw.head()

State raw shape: (104, 2)


,label,population
0,Alabama,NaN
1,Alabama,"5,024,279"
2,Alaska,NaN
3,Alaska,"733,391"
4,Arizona,NaN


### Extraction - County Population Data

In [6]:
# Extract county.csv from Azure Blob Storage
blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob='county.csv')
blob_data = blob_client.download_blob().readall()
county_raw = pd.read_csv(io.BytesIO(blob_data), encoding='utf-8-sig')
county_raw.columns = ['label', 'population']
print('County raw shape:', county_raw.shape)
county_raw.head()

County raw shape: (6444, 2)


,label,population
0,"Autauga County, Alabama",NaN
1,Estimate,"59,285"
2,"Baldwin County, Alabama",NaN
3,Estimate,"239,945"
4,"Barbour County, Alabama",NaN


### Extraction - ZIP to County/State Crosswalk

In [7]:
# Download ZIP-to-county-to-state crosswalk from public source
crosswalk_url = 'https://raw.githubusercontent.com/scpike/us-state-county-zip/master/geo-data.csv'
r = requests.get(crosswalk_url)
crosswalk_raw = pd.read_csv(io.StringIO(r.text), dtype=str)
print('Crosswalk raw shape:', crosswalk_raw.shape)
print('Crosswalk columns:', crosswalk_raw.columns.tolist())
crosswalk_raw.head()

Crosswalk raw shape: (33103, 6)
Crosswalk columns: ['state_fips', 'state', 'state_abbr', 'zipcode', 'county', 'city']


,state_fips,state,state_abbr,zipcode,county,city
0,1,Alabama,AL,35004,St. Clair,Acmar
1,1,Alabama,AL,35005,Jefferson,Adamsville
2,1,Alabama,AL,35006,Jefferson,Adger
3,1,Alabama,AL,35007,Shelby,Keystone
4,1,Alabama,AL,35010,Tallapoosa,New site


## Transformation

### Data Profiling

In [8]:
print('=== ZIP RAW ===')
print('Shape:', zip_raw.shape)
print(zip_raw.info())
print('Null counts:')
print(zip_raw.isnull().sum())

=== ZIP RAW ===
Shape: (67548, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67548 entries, 0 to 67547
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   label       67548 non-null  object
 1   population  33774 non-null  object
dtypes: object(2)
memory usage: 1.0+ MB
None
Null counts:
label             0
population    33774
dtype: int64


In [9]:
print('=== STATE RAW ===')
print('Shape:', state_raw.shape)
print(state_raw.info())
print('Null counts:')
print(state_raw.isnull().sum())

=== STATE RAW ===
Shape: (104, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   label       104 non-null    object
 1   population  52 non-null     object
dtypes: object(2)
memory usage: 1.8+ KB
None
Null counts:
label          0
population    52
dtype: int64


In [10]:
print('=== COUNTY RAW ===')
print('Shape:', county_raw.shape)
print(county_raw.info())
print('Null counts:')
print(county_raw.isnull().sum())

=== COUNTY RAW ===
Shape: (6444, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6444 entries, 0 to 6443
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   label       6444 non-null   object
 1   population  3222 non-null   object
dtypes: object(2)
memory usage: 100.8+ KB
None
Null counts:
label            0
population    3222
dtype: int64


In [11]:
print('=== CROSSWALK RAW ===')
print('Shape:', crosswalk_raw.shape)
print(crosswalk_raw.info())
print('Null counts:')
print(crosswalk_raw.isnull().sum())

=== CROSSWALK RAW ===
Shape: (33103, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33103 entries, 0 to 33102
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   state_fips  33103 non-null  object
 1   state       33103 non-null  object
 2   state_abbr  33103 non-null  object
 3   zipcode     33103 non-null  object
 4   county      33103 non-null  object
 5   city        33044 non-null  object
dtypes: object(6)
memory usage: 1.5+ MB
None
Null counts:
state_fips     0
state          0
state_abbr     0
zipcode        0
county         0
city          59
dtype: int64


### Data Cleaning

In [12]:
# Clean ZIP - keep only ZCTA5 rows that have a population value
zip_clean = zip_raw[zip_raw['label'].str.contains('ZCTA5', na=False)].copy()
zip_clean = zip_clean[zip_clean['population'].notna() & (zip_clean['population'] != '')]
zip_clean = zip_clean.dropna(subset=['population'])
print('ZIP after cleaning:', zip_clean.shape)
zip_clean.head()

ZIP after cleaning: (33774, 2)


,label,population
1,ZCTA5 00601,"17,242"
3,ZCTA5 00602,"37,548"
5,ZCTA5 00603,"49,804"
7,ZCTA5 00606,"5,009"
9,ZCTA5 00610,"25,731"


In [13]:
# Clean State - rows with \xa0 padding are data rows
state_clean = state_raw[state_raw['label'].str.startswith('\xa0')].copy()
state_clean = state_clean[state_clean['population'].notna() & (state_clean['population'] != '')]
state_clean = state_clean.dropna(subset=['population'])
print('State after cleaning:', state_clean.shape)
state_clean.head()

State after cleaning: (52, 2)


,label,population
1,Alabama,"5,024,279"
3,Alaska,"733,391"
5,Arizona,"7,151,502"
7,Arkansas,"3,011,524"
9,California,"39,538,223"


In [14]:
# Clean County - county name and population alternate rows
# Row N = county name (no population), Row N+1 = Estimate (has population)
# We pair them up using a loop
county_raw_reset = county_raw.reset_index(drop=True)
counties = []
for i in range(0, len(county_raw_reset) - 1, 2):
    name_row = str(county_raw_reset.iloc[i]['label']).strip()
    pop_row = county_raw_reset.iloc[i + 1]['population']
    if ',' in name_row and str(pop_row).strip() not in ['', 'nan']:
        counties.append({'label': name_row, 'population': str(pop_row).strip()})

county_clean = pd.DataFrame(counties)
print('County after cleaning:', county_clean.shape)
county_clean.head()

County after cleaning: (3222, 2)


,label,population
0,"Autauga County, Alabama","59,285"
1,"Baldwin County, Alabama","239,945"
2,"Barbour County, Alabama","24,757"
3,"Bibb County, Alabama","22,152"
4,"Blount County, Alabama","59,292"


### Data Reformatting

In [15]:
# Reformat ZIP
zip_clean['zip_code'] = zip_clean['label'].str.extract(r'(\d{5})')
zip_clean['zip_code'] = zip_clean['zip_code'].astype(str).str.zfill(5)
zip_clean['zip_population'] = zip_clean['population'].str.replace(',', '').astype(int)
zip_clean = zip_clean[['zip_code', 'zip_population']].reset_index(drop=True)
print('ZIP reformatted:', zip_clean.shape)
zip_clean.head()

ZIP reformatted: (33774, 2)


,zip_code,zip_population
0,00601,17242
1,00602,37548
2,00603,49804
3,00606,5009
4,00610,25731


In [16]:
# Reformat State
state_clean['state'] = state_clean['label'].str.replace('\xa0', '').str.strip()
state_clean['state_population'] = state_clean['population'].str.replace(',', '').astype(int)
state_clean = state_clean[['state', 'state_population']].reset_index(drop=True)
print('State reformatted:', state_clean.shape)
state_clean.head()

State reformatted: (52, 2)


,state,state_population
0,Alabama,5024279
1,Alaska,733391
2,Arizona,7151502
3,Arkansas,3011524
4,California,39538223


In [17]:
# Reformat County - split 'Autauga County, Alabama' into county and state
county_clean['county'] = county_clean['label'].apply(lambda x: x.rsplit(',', 1)[0].strip())
county_clean['state'] = county_clean['label'].apply(lambda x: x.rsplit(',', 1)[1].strip())
county_clean['county_population'] = county_clean['population'].str.replace(',', '').astype(int)
# Remove 'County' suffix to match crosswalk format
county_clean['county'] = county_clean['county'].str.replace(' County', '', regex=False).str.strip()
county_clean = county_clean[['county', 'state', 'county_population']].reset_index(drop=True)
print('County reformatted:', county_clean.shape)
county_clean.head()

County reformatted: (3222, 3)


,county,state,county_population
0,Autauga,Alabama,59285
1,Baldwin,Alabama,239945
2,Barbour,Alabama,24757
3,Bibb,Alabama,22152
4,Blount,Alabama,59292


In [18]:
# Reformat Crosswalk - pad zip codes and rename column
crosswalk = crosswalk_raw.copy()
crosswalk["zipcode"] = crosswalk["zipcode"].astype(str).str.zfill(5)
crosswalk = crosswalk.rename(columns={"zipcode": "zip_code"})
# Fix capitalization mismatch - crosswalk has e.g. "New york" but state file has "New York"
crosswalk["state"] = crosswalk["state"].str.title()
print("Crosswalk reformatted:", crosswalk.shape)
print("Columns:", crosswalk.columns.tolist())
crosswalk.head()

Crosswalk reformatted: (33103, 6)
Columns: ['state_fips', 'state', 'state_abbr', 'zip_code', 'county', 'city']


,state_fips,state,state_abbr,zip_code,county,city
0,1,Alabama,AL,35004,St. Clair,Acmar
1,1,Alabama,AL,35005,Jefferson,Adamsville
2,1,Alabama,AL,35006,Jefferson,Adger
3,1,Alabama,AL,35007,Shelby,Keystone
4,1,Alabama,AL,35010,Tallapoosa,New site


### Data Transformation - Build Dimension Table

In [19]:
# Build dim_geography
# Step 1: Join zip population to crosswalk to get county/state/city per zip
merged = zip_clean.merge(
    crosswalk[['zip_code', 'state', 'state_abbr', 'county', 'city']],
    on='zip_code', how='left'
)

# Step 2: Join state population
merged = merged.merge(state_clean, on='state', how='left')

# Step 3: Join county population
merged = merged.merge(county_clean, on=['county', 'state'], how='left')

# Step 4: Drop non-US territories (Puerto Rico etc - no state match)
merged = merged.dropna(subset=['state'])

# Step 5: Build final dim_geography with one row per zip code
dim_geography = merged[[
    'zip_code', 'city', 'county', 'state', 'state_abbr',
    'zip_population', 'county_population', 'state_population'
]].drop_duplicates(subset=['zip_code']).reset_index(drop=True)

# Add surrogate key
dim_geography.insert(0, 'geography_id', range(1, len(dim_geography) + 1))

print('dim_geography shape:', dim_geography.shape)
print('Null counts:')
print(dim_geography.isnull().sum())
dim_geography.head()

dim_geography shape: (31409, 9)
Null counts:
geography_id           0
zip_code               0
city                  26
county                 0
state                  0
state_abbr             0
zip_population         0
county_population    524
state_population      23
dtype: int64


,geography_id,zip_code,city,county,state,state_abbr,zip_population,county_population,state_population
0,1,01001,Agawam,Hampden,Massachusetts,MA,16984,462853.0,7029917.0
1,2,01002,Cushman,Hampshire,Massachusetts,MA,27558,156595.0,7029917.0
2,3,01005,Barre,Worcester,Massachusetts,MA,4900,861664.0,7029917.0
3,4,01007,Belchertown,Hampshire,Massachusetts,MA,15423,156595.0,7029917.0
4,5,01008,Blandford,Hampden,Massachusetts,MA,1317,462853.0,7029917.0


### Data Consolidation

In [20]:
# Final review before loading
print('=== dim_geography ===')
print('Shape:', dim_geography.shape)
print('Dtypes:')
print(dim_geography.dtypes)
print('Sample rows:')
dim_geography.head(10)

=== dim_geography ===
Shape: (31409, 9)
Dtypes:
geography_id           int64
zip_code              object
city                  object
county                object
state                 object
state_abbr            object
zip_population         int64
county_population    float64
state_population     float64
dtype: object
Sample rows:


,geography_id,zip_code,city,county,state,state_abbr,zip_population,county_population,state_population
0,1,01001,Agawam,Hampden,Massachusetts,MA,16984,462853.0,7029917.0
1,2,01002,Cushman,Hampshire,Massachusetts,MA,27558,156595.0,7029917.0
2,3,01005,Barre,Worcester,Massachusetts,MA,4900,861664.0,7029917.0
3,4,01007,Belchertown,Hampshire,Massachusetts,MA,15423,156595.0,7029917.0
4,5,01008,Blandford,Hampden,Massachusetts,MA,1317,462853.0,7029917.0
5,6,01010,Brimfield,Hampden,Massachusetts,MA,3711,462853.0,7029917.0
6,7,01011,Chester,Hampden,Massachusetts,MA,1201,462853.0,7029917.0
7,8,01012,Chesterfield,Hampshire,Massachusetts,MA,519,156595.0,7029917.0
8,9,01013,Chicopee,Hampden,Massachusetts,MA,23656,462853.0,7029917.0
9,10,01020,Chicopee,Hampden,Massachusetts,MA,29781,462853.0,7029917.0


## Loading

In [21]:
# Create dim_geography table in Snowflake
from sqlalchemy import text

create_table_sql = """
CREATE TABLE IF NOT EXISTS dim_geography (
    geography_id      INTEGER,
    zip_code          VARCHAR(10),
    city              VARCHAR(100),
    county            VARCHAR(100),
    state             VARCHAR(100),
    state_abbr        VARCHAR(5),
    zip_population    INTEGER,
    county_population INTEGER,
    state_population  INTEGER
)
"""

with engine.connect() as conn:
    conn.execute(text(create_table_sql))
    conn.commit()

print('Table created successfully')

Table created successfully


In [22]:
# Load dim_geography to Snowflake in chunks
chunksize = 5000
for i in range(0, dim_geography.shape[0], chunksize):
    chunk = dim_geography[i:i + chunksize]
    chunk.to_sql('dim_geography', engine, if_exists='append', index=False, method='multi')
    print(f'Loaded rows {i} to {i + len(chunk)}')

print('dim_geography loaded to Snowflake successfully!')

Loaded rows 0 to 5000
Loaded rows 5000 to 10000
Loaded rows 10000 to 15000
Loaded rows 15000 to 20000
Loaded rows 20000 to 25000
Loaded rows 25000 to 30000
Loaded rows 30000 to 31409
dim_geography loaded to Snowflake successfully!


In [23]:
# Verify load in Snowflake
result = pd.read_sql('SELECT COUNT(*) AS total FROM dim_geography', engine)
print('Total rows in Snowflake dim_geography:', result['total'][0])

# Preview loaded data
preview = pd.read_sql('SELECT * FROM dim_geography LIMIT 5', engine)
print(preview)

Total rows in Snowflake dim_geography: 188454
   geography_id zip_code       city   county  state state_abbr  \
0         25001    76012  Arlington  Tarrant  Texas         TX   
1         25002    76013  Arlington  Tarrant  Texas         TX   
2         25003    76014  Arlington  Tarrant  Texas         TX   
3         25004    76015  Arlington  Tarrant  Texas         TX   
4         25005    76016  Arlington  Tarrant  Texas         TX   

   zip_population  county_population  state_population  doctor_count_zip  \
0           27080          2135743.0        29145505.0               154   
1           36535          2135743.0        29145505.0                72   
2           35954          2135743.0        29145505.0                83   
3           17185          2135743.0        29145505.0               100   
4           31091          2135743.0        29145505.0                30   

   doctor_count_county  doctor_count_state  
0                 5499               85967  
1         